In [1]:
import pandas as pd
import os
import pyarrow
from pathlib import Path

#PROJECT_ROOT = Path(__file__).resolve().parents[1]
PROJECT_ROOT = Path.cwd().resolve().parents[0]
SILVER_DATASETS_DIR = PROJECT_ROOT / 'datasets' / 'silver'
BRONZE_DATASETS_DIR = PROJECT_ROOT / 'datasets' / 'bronze'

## Patch Summary Dimension

In [ ]:

patches_desc_df = pd.read_csv(BRONZE_DATASETS_DIR / 'patches/scraped_patch_notes.csv')

#patches_desc_df.info()
#patches_desc_df.head(5)


patches_desc_df['patch_start_date'] = pd.to_datetime(patches_desc_df['patch_date'], format='%m/%d/%Y', errors='coerce')
patches_desc_df = patches_desc_df.sort_values(by='patch_start_date', ascending=True)
patches_desc_df['patch_end_date'] = patches_desc_df['patch_start_date'].shift(-1)

patches_desc_df.head(5)

patches_dimension_dir = SILVER_DATASETS_DIR / 'patches_dimension.parquet'
patches_desc_df[['patch_number', 'patch_start_date', 'patch_end_date', 'patch_url']].to_parquet(patches_dimension_dir, engine='pyarrow', index=False)

In [ ]:
##Champions Changes Dimension

patches = pd.read_parquet(SILVER_DATASETS_DIR/'patches_dimension.parquet', engine='pyarrow')

if not os.path.exists(SILVER_DATASETS_DIR / 'champions_changes_fact.parquet'):
    champions_changes = pd.DataFrame(columns=['patch_number', 'champion_name', 'change_type'])
else:
    champions_changes = pd.read_parquet(SILVER_DATASETS_DIR/'champions_changes_dimension.parquet', engine='pyarrow')

for patch in patches.itertuples():
    if patch.patch_number not in champions_changes['patch_number'].values:
        source_file = BRONZE_DATASETS_DIR / f'patches/patch_highlights_champion_changes_{patch.patch_number}.csv'
        if os.path.exists(source_file):
            source_file_data = pd.read_csv(source_file)
            new_changes = pd.DataFrame(columns=['patch_number', 'champion_name', 'change_type'])
            new_changes['patch_number'] = source_file_data['Patch Number']
            new_changes['champion_name'] = source_file_data['Champion']
            new_changes['change_type'] = source_file_data['Change Type']
            champions_changes = pd.concat([champions_changes, new_changes], ignore_index=True)
        else:
            print(f"File {source_file} does not exist.")

champions_changes.to_parquet(SILVER_DATASETS_DIR / 'champions_changes_dimension.parquet', engine='pyarrow', index=False)


## Pro Matches Hist

In [ ]:
oe_file_dir = BRONZE_DATASETS_DIR / 'oe' / '2018_LoL_esports_match_data_from_OraclesElixir.csv'

oe_df = pd.read_csv(oe_file_dir)
match_summary_columns = ['date','side','position','playername','champion','gamelength','result','kills','deaths','assists',
                        'teamkills','teamdeaths','doublekills','triplekills','quadrakills','pentakills','damagetochampions',
                        'dpm','damageshare','damagetakenperminute','damagemitigatedperminute','damagetotowers','total cs',
                        'killsat10','assistsat10','deathsat10','csat10','opp_killsat10','opp_assistsat10','opp_deathsat10','opp_csat10',
                        'killsat15','assistsat15','deathsat15','csat15','opp_killsat15','opp_assistsat15','opp_deathsat15','opp_csat15',
                        'killsat20','assistsat20','deathsat20','csat20','opp_killsat20','opp_assistsat20','opp_deathsat20','opp_csat20',
                        'killsat25','assistsat25','deathsat25','csat25','opp_killsat25','opp_assistsat25','opp_deathsat25','opp_csat25']
ban_list_columns = ['date', 'gameid', 'side', 'ban1', 'ban2', 'ban3', 'ban4', 'ban5']

oe_df['date'] = pd.to_datetime(oe_df['date'], errors='coerce')
matches_summary_df = (oe_df.loc[:, match_summary_columns].rename(columns={'total cs': 'total_cs'}).copy().dropna(subset=['date','champion']))
matches_summary_df.to_parquet(SILVER_DATASETS_DIR / 'pro_matches_summary.parquet', engine='pyarrow', index=False)

ban_list_df = (oe_df.loc[:, ban_list_columns].copy().dropna(subset=['date','gameid','ban1']).drop_duplicates(subset=['date','gameid','side'], keep='first'))
ban_list_df.to_parquet(SILVER_DATASETS_DIR / 'pro_matches_ban_list.parquet', engine='pyarrow', index=False)


In [ ]:
from matches import get_patch_dates, get_last_match_date
from datetime import datetime

##Get first and last patch dates from the patches dimension
first_available_patch_date, last_available_patch_date = get_patch_dates()

##Get last match date from the procceded matches
last_processed_match_date = get_last_match_date()

years_list = list(range(first_available_patch_date.year, datetime.now().year + 1))

print(f"First available patch date: {first_available_patch_date}")
print(f"Last available patch date: {last_available_patch_date}")
print(f"Last processed match date: {last_processed_match_date}")
print(f"Years list: {years_list}")
print(f"Current year: {datetime.now().year}")

First available patch date: 2019-01-08 00:00:00
Last available patch date: 2026-08-11 00:00:00
Last processed match date: 2018-12-31 09:51:07
Years list: [2019, 2020, 2021, 2022, 2023, 2024, 2025]
Current year: 2026


In [2]:
from matches import upload_oe_matches

result = upload_oe_matches(rewrite=False)
print(f"Upload result: {result}")

c:\Users\Feragon\Documents\Portfolio\lol_patches\transformations\matches.py:48: DtypeWarning: Columns (0: url) have mixed types. Specify dtype option on import or set low_memory=False.
  oe_df = pd.read_csv(source_file_dir)
c:\Users\Feragon\Documents\Portfolio\lol_patches\transformations\matches.py:48: DtypeWarning: Columns (0: url) have mixed types. Specify dtype option on import or set low_memory=False.
  oe_df = pd.read_csv(source_file_dir)
c:\Users\Feragon\Documents\Portfolio\lol_patches\transformations\matches.py:48: DtypeWarning: Columns (0: url) have mixed types. Specify dtype option on import or set low_memory=False.
  oe_df = pd.read_csv(source_file_dir)
c:\Users\Feragon\Documents\Portfolio\lol_patches\transformations\matches.py:48: DtypeWarning: Columns (0: url, 1: split) have mixed types. Specify dtype option on import or set low_memory=False.
  oe_df = pd.read_csv(source_file_dir)
c:\Users\Feragon\Documents\Portfolio\lol_patches\transformations\matches.py:48: DtypeWarning: 

Upload result: True


In [5]:
import duckdb

duckdb_conn = duckdb.connect(database=':memory:')
duckdb_conn.execute(f"CREATE TABLE pro_matches_summary AS SELECT * FROM read_parquet('{SILVER_DATASETS_DIR}/pro_matches_summary.parquet')")
duckdb_conn.execute("SELECT year(date) as match_year, COUNT(*) as match_count FROM pro_matches_summary GROUP BY match_year ORDER BY match_year").fetchall()

[(2019, 81270),
 (2020, 97470),
 (2021, 123020),
 (2022, 125260),
 (2023, 111060),
 (2024, 101920),
 (2025, 100380),
 (2026, 59820)]

# Get  Solo Queue matches

In [29]:


#check on json structure
sq = pd.read_json(BRONZE_DATASETS_DIR / 'riot_api' / '2026-07-20_match_info_results.json', lines=True)
sq.head(5)

,metadata,info
0,"{'dataVersion': '2', 'matchId': 'NA1_546386260...","{'endOfGameResult': 'GameComplete', 'gameCreat..."
1,"{'dataVersion': '2', 'matchId': 'NA1_546382565...","{'endOfGameResult': 'GameComplete', 'gameCreat..."
2,"{'dataVersion': '2', 'matchId': 'NA1_547805591...","{'endOfGameResult': 'GameComplete', 'gameCreat..."
3,"{'dataVersion': '2', 'matchId': 'NA1_547803747...","{'endOfGameResult': 'GameComplete', 'gameCreat..."
4,"{'dataVersion': '2', 'matchId': 'NA1_547796833...","{'endOfGameResult': 'GameComplete', 'gameCreat..."


In [30]:
# sq_info = sq.iloc[0].info
# df = pd.DataFrame(sq_info)
# df = pd.DataFrame([sq_info])
# df = pd.json_normalize(sq_info)
# df = pd.json_normalize(sq.iloc[0])
# df = pd.DataFrame(sq.iloc[0])
# df = pd.DataFrame([sq.iloc[0]])
# df = pd.json_normalize(sq)

# base_df = pd.DataFrame({
#     'match_id' : sq['metadata'].apply(lambda x: x['matchId']),
#     'match_info' : sq['info'].apply(lambda x: x)
# })

# base_df.head(5)

# for match in base_df.itertuples():
#     match_id = match.match_id
#     match_info = match.match_info
#     df = pd.json_normalize(match_info)
#     df['match_id'] = match_id
#     if 'matches_df' not in locals():
#         matches_df = df
#     else:
#         matches_df = pd.concat([matches_df, df], ignore_index=True)

match_ids = sq['metadata'].map(lambda x: x['matchId'])
matches_df = pd.json_normalize(sq['info'].tolist(), sep = "_")
matches_df.insert(0, 'match_id', match_ids)
matches_df.to_csv(SILVER_DATASETS_DIR / 'solo_queue_matches.csv', index=False)

In [46]:
matches_df.columns
#matches_df.head(5)
# participants = pd.json_normalize(matches_df.iloc[0]['participants'])
# participants.head(5)

# teams = pd.json_normalize(matches_df.iloc[0]['teams'])
# teams.head(5)




Index(['match_id', 'endOfGameResult', 'gameCreation', 'gameDuration',
       'gameEndTimestamp', 'gameId', 'gameMode', 'gameName',
       'gameStartTimestamp', 'gameType', 'gameVersion', 'mapId',
       'participants', 'platformId', 'queueId', 'teams', 'tournamentCode'],
      dtype='str')

In [ ]:
## Banned Champions List

teams = pd.json_normalize(matches_df.iloc[0]['teams'])
teams.head(5)

bans = pd.json_normalize(teams.iloc[0]['bans'])
banned_champions_id_list = bans['championId'].tolist()
print(f"Banned Champions ID List: {banned_champions_id_list}")


Banned Champions ID List: [107, 51, 201, 887, 134]


In [64]:
## List of Champions in the Match

participants = pd.json_normalize(matches_df.iloc[0]['participants'])
selected_champions = participants['championName'].tolist()
print(f"Selected Champions List: {selected_champions}")


Selected Champions List: ['Renekton', 'Evelynn', 'Heimerdinger', 'Kaisa', 'Janna', 'Gangplank', 'Nunu', 'TwistedFate', 'Jhin', 'Pyke']


In [ ]:
## Complete Champion Info

participants = pd.json_normalize(matches_df.iloc[1]['participants'])

columns = ['championName',
           'teamId', ##Side 100 - Blue / 200 - Red
           'teamPosition',
           'win', #Result
           'gameEndedInSurrender',
           'kills','deaths','assists',
           'totalDamageDealtToChampions', 'totalDamageTaken',
           'doubleKills','tripleKills','quadraKills','pentaKills',
           'longestTimeSpentLiving', 'largestKillingSpree', 'largestMultiKill',
           'totalMinionsKilled'
           ]

champions_info = participants[columns]
champions_info['teamKills'] = champions_info.groupby('teamId')['kills'].transform('sum')
champions_info['teamDeaths'] = champions_info.groupby('teamId')['deaths'].transform('sum')
champions_info['teamAssists'] = champions_info.groupby('teamId')['assists'].transform('sum')

champions_info.head(10)

,championName,teamId,teamPosition,win,gameEndedInSurrender,kills,deaths,assists,totalDamageDealtToChampions,totalDamageTaken,...,tripleKills,quadraKills,pentaKills,longestTimeSpentLiving,largestKillingSpree,largestMultiKill,totalMinionsKilled,teamKills,teamDeaths,teamAssists
0,Aatrox,100,TOP,True,False,19,2,14,42035,41077,...,1,0,0,1434,19,3,204,55,24,87
1,Jayce,100,JUNGLE,True,False,14,4,15,24546,18826,...,0,0,0,779,9,2,47,55,24,87
2,Velkoz,100,MIDDLE,True,False,7,8,12,23698,20637,...,0,0,0,239,2,2,166,55,24,87
3,MissFortune,100,BOTTOM,True,False,11,6,23,37080,17551,...,1,0,0,517,8,3,151,55,24,87
4,Nautilus,100,UTILITY,True,False,4,4,23,14838,20589,...,0,0,0,912,2,1,31,55,24,87
5,DrMundo,200,TOP,False,False,1,10,6,19057,44893,...,0,0,0,234,0,1,161,24,55,40
6,Sylas,200,JUNGLE,False,False,3,13,9,14134,43034,...,0,0,0,332,0,1,6,24,55,40
7,Akshan,200,MIDDLE,False,False,8,9,10,23009,23505,...,0,0,0,430,2,2,189,24,55,40
8,Samira,200,BOTTOM,False,False,9,11,6,30007,25663,...,1,1,0,256,2,4,218,24,55,40
9,Rell,200,UTILITY,False,False,3,12,9,9307,29804,...,0,0,0,219,0,1,25,24,55,40


In [63]:
## Match Basic Information
patch = '.'.join(matches_df.iloc[0]['gameVersion'].split('.')[:2])
print(f"Match Patch Version: {patch}")

gameCreationTime = pd.to_datetime(matches_df.iloc[0]['gameCreation'], unit='ms')
print(f"Match Creation Time: {gameCreationTime}")

gameDuration = pd.to_timedelta(matches_df.iloc[0]['gameDuration'], unit='s')
print(f"Match Duration: {gameDuration}")

gameResult = matches_df.iloc[0]['endOfGameResult']
print(f"Match Result: {gameResult}")

Match Patch Version: 16.1
Match Creation Time: 2026-01-14 02:42:34.424000
Match Duration: 0 days 00:33:34
Match Result: GameComplete


TODO: 
*Function to unify API info into a parquet similar to OE result.
*Include Patch into OE result
